In [1]:
import sympy as sp
from IPython.display import display, Math

# 1. Configuración gráfica
sp.init_printing(use_latex='mathjax')

# 2. Definición de variables universales
t, w, s = sp.symbols('t w s', real=True)
a, alpha = sp.symbols('a alpha', positive=True, real=True)
j = sp.I

def asistente_teoremas_y_tablas(f_t=None, F_w=None):
    print("==================================================")
    print(" ASISTENTE MAESTRO: TEOREMAS Y TABLAS")
    print("==================================================\n")
    
    # ----------------------------------------------------
    # MODO 1: TRANSFORMADA DIRECTA (Tiempo a Frecuencia)
    # ----------------------------------------------------
    if f_t is not None:
        print("1. Función original en el tiempo f(t):")
        display(f_t)
        
        # Separamos factores para buscar corrimientos (El Sistema Experto)
        factores = f_t.as_ordered_factors()
        modulador = None
        base_t = 1
        
        for factor in factores:
            if factor.is_Pow and factor.base == sp.E and factor.exp.has(j):
                modulador = factor
            else:
                base_t *= factor
                
        if modulador is not None:
            print("\n---> ¡CORRIMIENTO EN FRECUENCIA DETECTADO! <---")
            display(Math(rf"\text{{Base: }} f(t) = {sp.latex(base_t)}"))
            display(Math(rf"\text{{Modulador: }} {sp.latex(modulador)}"))
            
            print("\n2. Transformada de la Base F(w):")
            try:
                F_laplace = sp.laplace_transform(base_t, t, s)[0]
                F_base_w = F_laplace.subs(s, j*w)
                display(F_base_w)
            except:
                print("Error: La base no está en tablas.")
                return
            
            print("\n3. Extrayendo corrimiento (w0) y aplicando Teorema:")
            w0_detectado = sp.simplify(modulador.exp / (j * t))
            display(Math(rf"\omega_0 = {sp.latex(w0_detectado)}"))
            
            F_final = F_base_w.subs(w, w - w0_detectado)
            
        else:
            print("\n---> TRANSFORMADA DIRECTA POR TABLA <---")
            try:
                F_laplace = sp.laplace_transform(base_t, t, s)[0]
                F_final = F_laplace.subs(s, j*w)
            except:
                print("Error: La función no está en tablas.")
                return
            
        print("\n==================================================")
        print(" RESULTADO FINAL F(w):")
        display(sp.simplify(F_final))
        print("==================================================")

# ----------------------------------------------------
    # MODO 2: TRANSFORMADA INVERSA (Mejorada con Detección)
    # ----------------------------------------------------
    elif F_w is not None:
        print("1. Función espectral ingresada F(w):")
        display(F_w)
        
        print("\n2. Transformada Inversa f(t) resultante (para t > 0):")
        try:
            # 1. Buscamos si hay un corrimiento (w - w0)
            w0_detectado = 0
            terminos_sumas = [arg for arg in sp.preorder_traversal(F_w) if isinstance(arg, sp.Add) and arg.has(w)]
            for expresion in terminos_sumas:
                try:
                    w0_temp = sp.solve(w - sp.Symbol('w0') - expresion, sp.Symbol('w0'))
                    if w0_temp and not w0_temp[0].has(w):
                        # EL BUG ESTABA AQUÍ: Había un signo '-' antes de w0_temp[0]
                        w0_detectado = sp.simplify(w0_temp[0]) 
                        break
                    if expresion.has(j*w):
                        w0_temp = sp.solve(j*w - j*sp.Symbol('w0') - expresion, sp.Symbol('w0'))
                        if w0_temp and not w0_temp[0].has(w):
                            # EL BUG ESTABA AQUÍ TAMBIÉN
                            w0_detectado = sp.simplify(w0_temp[0]) 
                            break
                except:
                    continue
                    
            # 2. Quitamos el corrimiento para dejar la base "limpia"
            F_base_w = sp.simplify(F_w.subs(w, w + w0_detectado))
            
            # 3. Aplicamos Laplace Inverso a la base limpia (Garantiza senos/cosenos perfectos)
            F_s = F_base_w.subs(j * w, s).subs(w, s / j)
            f_base_t = sp.inverse_laplace_transform(F_s, s, t).subs(sp.Heaviside(t), 1)
            
            # 4. Parche hiperbólico (por si acaso quedaron hiperbólicos residuales)
            f_base_limpia = f_base_t.replace(sp.sinh, lambda arg: (sp.exp(arg) - sp.exp(-arg))/2).replace(sp.cosh, lambda arg: (sp.exp(arg) + sp.exp(-arg))/2)
            
            # 5. Le devolvemos su corrimiento multiplicando por e^(j*w0*t) al final
            f_final = sp.simplify(f_base_limpia) * sp.exp(j * w0_detectado * t)
            
            display(f_final)
            print("\n(Calculado vía extracción de base + Laplace inverso)")
        except Exception as e:
            print(f"SymPy no encontró esta inversa en su tabla interna. Error: {e}")
            
        print("==================================================")

# ==========================================
# 📝 ZONA DE CONFIGURACIÓN
# ==========================================

# MODO DIRECTO: Mete tu f(t) aquí (Ej. Problema 2b)
mi_f_tiempo = sp.exp(-j * alpha * t) * (1 - t**2)

# MODO INVERSO: Mete tu F(w) aquí (Ej. Problema 3)
mi_F_frecuencia = (3+j*w)/(109-w**2+6*j*w)

# --- EJECUCIÓN ---
# Descomenta la línea que necesites usar

# Para hacer el Problema 2b (Directa):
asistente_teoremas_y_tablas(f_t=None, F_w=mi_F_frecuencia)

# Para hacer el Problema 3 (Inversa):
# asistente_teoremas_y_tablas(f_t=None, F_w=mi_F_frecuencia)

 ASISTENTE MAESTRO: TEOREMAS Y TABLAS

1. Función espectral ingresada F(w):


     ⅈ⋅w + 3      
──────────────────
   2              
- w  + 6⋅ⅈ⋅w + 109


2. Transformada Inversa f(t) resultante (para t > 0):


 -3⋅t          
ℯ    ⋅cos(10⋅t)


(Calculado vía extracción de base + Laplace inverso)
